In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, KFold
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


| Column | Description |
|--------|-------------|
| **CRIM** | Per capita crime rate by town |
| **ZN** | Proportion of residential land zoned for lots over 25,000 sq. ft. |
| **INDUS** | Proportion of non-retail business acres per town |
| **CHAS** | Charles River dummy variable (= 1 if tract bounds river; 0 otherwise) |
| **NOX** | Nitric oxides concentration (parts per 10 million) |
| **RM** | Average number of rooms per dwelling |
| **AGE** | Proportion of owner-occupied units built prior to 1940 |
| **DIS** | Weighted distances to five Boston employment centres |
| **RAD** | Index of accessibility to radial highways |
| **TAX** | Full-value property-tax rate per $10,000 |
| **PTRATIO** | Pupil-teacher ratio by town |
| **B** | 1000(Bk - 0.63)^2 where Bk is the proportion of [people of African American descent] by town |
| **LSTAT** | Percentage of lower status of the population |
| **MEDV** | **Target variable:** Median value of owner-occupied homes in $1000's |

In [3]:
# https://www.kaggle.com/datasets/altavish/boston-housing-dataset
data = pd.read_csv('../datasets/HousingData.csv')
data.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,NaN,36.2


In [4]:
data.shape

(506, 14)

In [17]:
# data split
X = data.drop('MEDV', axis=1)
y = data['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# preprocessing
numeric_features = X.columns.tolist()
preprocessor = ColumnTransformer([
    ('imputer', SimpleImputer(), numeric_features) # fill missing data
])

# pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# hyperparameter tuning (search space)
param_dist = {
    'preprocessor__imputer__strategy': ['mean', 'median', 'most_frequent'], 
    'rf__n_estimators': [50, 100],
    'rf__max_depth': [5, 10, 20, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', None]
}

# cv
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# model based on best estimators
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)


# finding best params (hyperparameter tuning) and train final model/fit
random_search.fit(X_train, y_train)

# model (best model)
model = random_search.best_estimator_

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)

print(f"Best params: {random_search.best_estimator_}")
print(f"Best CV score (on training): {random_search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('imputer', SimpleImputer(),
                                                  ['CRIM', 'ZN', 'INDUS',
                                                   'CHAS', 'NOX', 'RM', 'AGE',
                                                   'DIS', 'RAD', 'TAX',
                                                   'PTRATIO', 'B',
                                                   'LSTAT'])])),
                ('scaler', StandardScaler()),
                ('rf',
                 RandomForestRegressor(max_depth=10, max_features='log2',
                                       n_estimators=50, n_jobs=-1,
                                       random_state=42))])
Best CV score (on training): 0.8294


In [18]:
# performance metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred)
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred)

print("\n" + "="*50)
print(f"Model Performance:")
print("="*50)

print(f"\n📊 R²")
print(f"   • Model explains {r2_test*100:.1f}% of the variation in house price of unit area")
print(f"   • Only {100-r2_test*100:.1f}% of variation is unexplained (due to other factors)")
print(f"Training R²: {r2_train:.4f}")
print(f"Test R²: {r2_test:.4f}")

if abs(r2_train - r2_test) < 0.05:
    print("✅ Good: Training and testing R² are similar - no overfitting")
else:
    print(f"⚠️  Warning: Difference of {abs(r2_train - r2_test):.3f} between training and testing R²")

print()

print(f"\n📊 Cross Validation:")
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"Cross-validation R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print()

print(f"\n📊 MSE:")
print(f"Training MSE: {mse_train:.2f}")
print(f"Test MSE: {mse_test:.2f}")

print()

print(f"\n📊 RMSE")
print(f"   • Predictions are off by ±{rmse_test:.2f} on average")
print(f"   • In other words, 68% of predictions fall within {rmse_test:.2f} of actual value")
print(f"   • 95% of predictions fall within {rmse_test*2:.2f} of actual value")
print(f"Training RMSE: {rmse_train:.2f}")
print(f"Testing RMSE: {rmse_test:.2f}")

print()

print(f"\n📊 MAE:")
print(f"Training MAE: {mae_train:.2f}")
print(f"Testing MAE: {mae_test:.2f}")


Model Performance:

📊 R²
   • Model explains 84.8% of the variation in house price of unit area
   • Only 15.2% of variation is unexplained (due to other factors)
Training R²: 0.9733
Test R²: 0.8481
⚠️  Warning: Difference of 0.125 between training and testing R²


📊 Cross Validation:
Cross-validation R²: 0.604 (+/- 0.235)


📊 MSE:
Training MSE: 2.32
Test MSE: 11.14


📊 RMSE
   • Predictions are off by ±3.34 on average
   • In other words, 68% of predictions fall within 3.34 of actual value
   • 95% of predictions fall within 6.68 of actual value
Training RMSE: 1.52
Testing RMSE: 3.34


📊 MAE:
Training MAE: 1.11
Testing MAE: 2.12


In [11]:
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.named_steps['rf'].feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance)



FEATURE IMPORTANCE
    Feature  Importance
5        RM    0.297323
12    LSTAT    0.220157
4       NOX    0.081297
2     INDUS    0.075638
7       DIS    0.072534
0      CRIM    0.065942
10  PTRATIO    0.060255
9       TAX    0.035508
6       AGE    0.031021
11        B    0.025921
8       RAD    0.013427
3      CHAS    0.011341
1        ZN    0.009637


### tuning hyperparameters to avoid overfiting

In [21]:
# More aggressive regularization (result less diff between train and test but not enough)
param_dist = {
    'preprocessor__imputer__strategy': ['mean', 'median'],
    'rf__n_estimators': [100, 200, 300],        # more trees
    'rf__max_depth': [5, 10],                   # Limit depth
    'rf__min_samples_split': [10, 20, 50],      # higher values
    'rf__min_samples_leaf': [5, 10, 20],        # higher values
    'rf__max_features': ['sqrt', 'log2'],       # Limit features
    'rf__max_samples': [0.5, 0.7, 0.9]          # Use subset of data
}

# cv
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# model based on best estimators
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)


# finding best params (hyperparameter tuning) and train final model/fit
random_search.fit(X_train, y_train)

# model (best model)
model = random_search.best_estimator_

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)

print(f"Best params: {random_search.best_estimator_}")
print(f"Best CV score (on training): {random_search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('imputer', SimpleImputer(),
                                                  ['CRIM', 'ZN', 'INDUS',
                                                   'CHAS', 'NOX', 'RM', 'AGE',
                                                   'DIS', 'RAD', 'TAX',
                                                   'PTRATIO', 'B',
                                                   'LSTAT'])])),
                ('scaler', StandardScaler()),
                ('rf',
                 RandomForestRegressor(max_depth=10, max_features='log2',
                                       max_samples=0.9, min_samples_leaf=5,
                                       min_samples_split=10, n_estimators=200,
                                       n_jobs=-1, random_state=42))])
Best CV score (on training): 0.7794


In [22]:
# performance metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred)
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred)

print("\n" + "="*50)
print(f"Model Performance:")
print("="*50)

print(f"\n📊 R²")
print(f"   • Model explains {r2_test*100:.1f}% of the variation in house price of unit area")
print(f"   • Only {100-r2_test*100:.1f}% of variation is unexplained (due to other factors)")
print(f"Training R²: {r2_train:.4f}")
print(f"Test R²: {r2_test:.4f}")

if abs(r2_train - r2_test) < 0.05:
    print("✅ Good: Training and testing R² are similar - no overfitting")
else:
    print(f"⚠️  Warning: Difference of {abs(r2_train - r2_test):.3f} between training and testing R²")

print()

print(f"\n📊 Cross Validation:")
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"Cross-validation R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print()

print(f"\n📊 MSE:")
print(f"Training MSE: {mse_train:.2f}")
print(f"Test MSE: {mse_test:.2f}")

print()

print(f"\n📊 RMSE")
print(f"   • Predictions are off by ±{rmse_test:.2f} on average")
print(f"   • In other words, 68% of predictions fall within {rmse_test:.2f} of actual value")
print(f"   • 95% of predictions fall within {rmse_test*2:.2f} of actual value")
print(f"Training RMSE: {rmse_train:.2f}")
print(f"Testing RMSE: {rmse_test:.2f}")

print()

print(f"\n📊 MAE:")
print(f"Training MAE: {mae_train:.2f}")
print(f"Testing MAE: {mae_test:.2f}")


Model Performance:

📊 R²
   • Model explains 79.8% of the variation in house price of unit area
   • Only 20.2% of variation is unexplained (due to other factors)
Training R²: 0.8899
Test R²: 0.7981
⚠️  Warning: Difference of 0.092 between training and testing R²


📊 Cross Validation:
Cross-validation R²: 0.583 (+/- 0.181)


📊 MSE:
Training MSE: 9.56
Test MSE: 14.80


📊 RMSE
   • Predictions are off by ±3.85 on average
   • In other words, 68% of predictions fall within 3.85 of actual value
   • 95% of predictions fall within 7.70 of actual value
Training RMSE: 3.09
Testing RMSE: 3.85


📊 MAE:
Training MAE: 1.99
Testing MAE: 2.19


### making model more simple to avoid overfiting

In [26]:
# Simpler model (less prone to overfitting) and cv with more folds
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        n_estimators=100,
        max_depth=10,              # Limit depth
        min_samples_split=10,      # More samples to split
        min_samples_leaf=5,        # More samples per leaf
        max_features='sqrt',       # Limit features per tree
        max_samples=0.8,           # Use 80% of training data per tree
        bootstrap=True
    ))
])

param_dist = {
    'preprocessor__imputer__strategy': ['mean', 'median'],
    'rf__n_estimators': [100, 200, 300],        # more trees
    'rf__max_depth': [5, 10],                   # Limit depth
    'rf__min_samples_split': [10, 20, 50],      # higher values
    'rf__min_samples_leaf': [5, 10, 20],        # higher values
    'rf__max_features': ['sqrt', 'log2'],       # Limit features
    'rf__max_samples': [0.5, 0.7, 0.9]          # Use subset of data
}

# cv
cv = KFold(n_splits=10, shuffle=True, random_state=42)

# model based on best estimators
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)


# finding best params (hyperparameter tuning) and train final model/fit
random_search.fit(X_train, y_train)

# model (best model)
model = random_search.best_estimator_

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)

print(f"Best params: {random_search.best_estimator_}")
print(f"Best CV score (on training): {random_search.best_score_:.4f}")                    

Fitting 10 folds for each of 50 candidates, totalling 500 fits
Best params: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('imputer',
                                                  SimpleImputer(strategy='median'),
                                                  ['CRIM', 'ZN', 'INDUS',
                                                   'CHAS', 'NOX', 'RM', 'AGE',
                                                   'DIS', 'RAD', 'TAX',
                                                   'PTRATIO', 'B',
                                                   'LSTAT'])])),
                ('scaler', StandardScaler()),
                ('rf',
                 RandomForestRegressor(max_depth=10, max_features='log2',
                                       max_samples=0.9, min_samples_leaf=5,
                                       min_samples_split=10, n_estimators=200,
                                       n_jobs=-1, random_state=42))])
Best CV score (on tra

In [27]:
# performance metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred)
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred)
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred)

print("\n" + "="*50)
print(f"Model Performance:")
print("="*50)

print(f"\n📊 R²")
print(f"   • Model explains {r2_test*100:.1f}% of the variation in house price of unit area")
print(f"   • Only {100-r2_test*100:.1f}% of variation is unexplained (due to other factors)")
print(f"Training R²: {r2_train:.4f}")
print(f"Test R²: {r2_test:.4f}")

if abs(r2_train - r2_test) < 0.05:
    print("✅ Good: Training and testing R² are similar - no overfitting")
else:
    print(f"⚠️  Warning: Difference of {abs(r2_train - r2_test):.3f} between training and testing R²")

print()

print(f"\n📊 Cross Validation:")
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"Cross-validation R²: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

print()

print(f"\n📊 MSE:")
print(f"Training MSE: {mse_train:.2f}")
print(f"Test MSE: {mse_test:.2f}")

print()

print(f"\n📊 RMSE")
print(f"   • Predictions are off by ±{rmse_test:.2f} on average")
print(f"   • In other words, 68% of predictions fall within {rmse_test:.2f} of actual value")
print(f"   • 95% of predictions fall within {rmse_test*2:.2f} of actual value")
print(f"Training RMSE: {rmse_train:.2f}")
print(f"Testing RMSE: {rmse_test:.2f}")

print()

print(f"\n📊 MAE:")
print(f"Training MAE: {mae_train:.2f}")
print(f"Testing MAE: {mae_test:.2f}")


Model Performance:

📊 R²
   • Model explains 79.8% of the variation in house price of unit area
   • Only 20.2% of variation is unexplained (due to other factors)
Training R²: 0.8890
Test R²: 0.7980
⚠️  Warning: Difference of 0.091 between training and testing R²


📊 Cross Validation:
Cross-validation R²: 0.579 (+/- 0.174)


📊 MSE:
Training MSE: 9.64
Test MSE: 14.81


📊 RMSE
   • Predictions are off by ±3.85 on average
   • In other words, 68% of predictions fall within 3.85 of actual value
   • 95% of predictions fall within 7.70 of actual value
Training RMSE: 3.11
Testing RMSE: 3.85


📊 MAE:
Training MAE: 1.99
Testing MAE: 2.18


In [28]:
# seems not working so maybe using xgboost for this data in future to compare results